# 🦀 Day 6: Docker — Containerizing a Rust Web App

---

> **Note:** This notebook is primarily markdown/educational. Docker commands run in your terminal, not inside EvCxR. We'll cover concepts and show Dockerfile/compose examples.

## What is Docker and Why Containerize?

**Docker** packages your application and its dependencies into a **container** — a lightweight, portable unit that runs consistently anywhere.

**Why containerize a Rust web app?**
- **Consistency** — "works on my machine" becomes "works everywhere"
- **Isolation** — app runs in its own environment, no conflicts with host
- **Deployment** — push image to registry, pull and run on any server
- **Orchestration** — Kubernetes, Docker Swarm, ECS all expect containers

## Dockerfile Basics

| Instruction | Purpose |
|-------------|----------|
| `FROM` | Base image (e.g., `rust:1.75`, `debian:slim`)
| `RUN` | Execute a command during build
| `COPY` | Copy files from host into image
| `WORKDIR` | Set working directory
| `CMD` | Default command when container runs
| `EXPOSE` | Document which port the app listens on
| `ENV` | Set environment variables

## Simple Single-Stage Dockerfile for Rust

```dockerfile
FROM rust:1.75
WORKDIR /app
COPY . .
RUN cargo build --release
EXPOSE 3000
CMD ["./target/release/my-app"]
```

**Problem:** Final image is huge (~1.5GB+) because it includes the full Rust toolchain. We only need the compiled binary!

## Multi-Stage Dockerfile for Rust

**Stage 1:** Build the binary with full Rust toolchain  
**Stage 2:** Copy only the binary into a minimal runtime image

```dockerfile
# Build stage
FROM rust:1.75-slim as builder
WORKDIR /app
COPY Cargo.toml Cargo.lock ./
COPY src ./src
RUN cargo build --release

# Runtime stage
FROM debian:bookworm-slim
RUN apt-get update && apt-get install -y ca-certificates && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY --from=builder /app/target/release/my-app .
EXPOSE 3000
CMD ["./my-app"]
```

Final image: ~50–100MB instead of 1.5GB!

## Using `rust:slim` and `debian:slim`

- **`rust:slim`** — Smaller Rust image for build stage (~600MB vs ~1.5GB)
- **`debian:bookworm-slim`** — Minimal Debian for runtime (~80MB)
- **`alpine`** — Even smaller but can have musl/glibc compatibility issues with some Rust crates

## `.dockerignore` File

Prevent unnecessary files from being sent to the Docker daemon (faster builds, smaller context):

```
target/
.git/
.gitignore
*.md
.env
.env.*
Dockerfile
docker-compose*.yml
```

## Docker Commands

```bash
# Build image
docker build -t my-rust-app:latest .

# Run container
docker run -p 3000:3000 --env PORT=3000 my-rust-app:latest

# Run in background
docker run -d -p 3000:3000 --name my-app my-rust-app:latest

# View logs
docker logs my-app

# Stop and remove
docker stop my-app && docker rm my-app
```

## Environment Variables in Docker

```bash
# Pass single var
docker run -e DATABASE_URL=postgres://localhost/db my-app

# Pass env file
docker run --env-file .env my-app

# In docker-compose
# environment:
#   - DATABASE_URL=postgres://db:5432/mydb
# env_file:
#   - .env
```

## Docker Compose for Multi-Service Setup

Run app + database (and more) together with one command:

```yaml
version: "3.8"

services:
  app:
    build: .
    ports:
      - "3000:3000"
    environment:
      - DATABASE_URL=postgres://postgres:secret@db:5432/mydb
    depends_on:
      db:
        condition: service_healthy

  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: secret
      POSTGRES_DB: mydb
    volumes:
      - pgdata:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres"]
      interval: 5s
      timeout: 5s
      retries: 5

volumes:
  pgdata:
```

## Health Checks

In Dockerfile:

```dockerfile
HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 \
  CMD curl -f http://localhost:3000/health || exit 1
```

In docker-compose:

```yaml
healthcheck:
  test: ["CMD", "curl", "-f", "http://localhost:3000/health"]
  interval: 30s
  timeout: 3s
  retries: 3
```

## Writing Dockerfile Content Programmatically

You can generate Dockerfile content from Rust (e.g., for tooling or templates):

In [ ]:
fn generate_dockerfile(app_name: &str, rust_version: &str) -> String {
    format!(r#"
# Build stage
FROM rust:{} as builder
WORKDIR /app
COPY . .
RUN cargo build --release

# Runtime stage
FROM debian:bookworm-slim
RUN apt-get update && apt-get install -y ca-certificates && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY --from=builder /app/target/release/{} .
EXPOSE 3000
CMD ["./{}"]
"#,
        rust_version,
        app_name,
        app_name
    )
}

let dockerfile = generate_dockerfile("my-rust-app", "1.75-slim");
println!("{}", dockerfile);

## 🏋️ Exercise

Write a `docker-compose.yml` for a Rust web app + PostgreSQL. Include:
- App service that builds from `./` and exposes port 3000
- PostgreSQL service with user, password, and database
- `depends_on` with health check so app waits for DB
- `DATABASE_URL` env var for the app pointing to the db service

In [ ]:
// Write your docker-compose.yml content as a string (for practice)
// In a real project, save this to docker-compose.yml

let docker_compose = r#"
# YOUR CODE HERE — write the full docker-compose.yml
# version: "3.8"
# services:
#   app: ...
#   db: ...
"#;

println!("{}", docker_compose);

## 🎯 Key Takeaways

1. **Docker** — packages app + deps into portable containers
2. **Multi-stage builds** — build with Rust, run with minimal Debian (small images)
3. **`.dockerignore`** — exclude `target/`, `.git`, etc. for faster builds
4. **`docker build` / `docker run`** — build images, run containers
5. **Docker Compose** — run app + database + more with one command
6. **Health checks** — let orchestrators know when the app is ready

---

**Tomorrow:** Mini-project — Full-stack URL shortener 🔗